# M2 v3 — Fase 2 con la base descongelada al 100 %

**TFM · HAM10000** · Experimento `m2-entrenamiento-v3` · Rama `exp/m2-v3-descongelado-100`

Un solo cambio respecto a la v2: en la fase 2 se descongela toda la base. La BatchNormalization sigue congelada. No toca los runs ni los alias de la v1 y de la v2.

Una semilla y las tres arquitecturas. `PILOTO = True` entrena solo ResNet50, para medir memoria y tiempo. Si Colab se desconecta, volver a ejecutar: una arquitectura en `estado=final` no se repite. Checkpoints en Drive, `checkpoints-v3/m2-<arch>-s<semilla>/`, incluido el mejor de la fase 1.

No evalúa el conjunto de prueba y no mueve alias.


## 0. Parámetros

La semilla y el piloto se eligen solo aquí.

In [ ]:
BRANCH = "exp/m2-v3-descongelado-100"
AUTOR = "alex"          # alex | adrian | christian
SEED = 42               # Alex 42, Adrián 123, Christian 2024
PILOTO = False          # True: solo ResNet50, la más pesada
ARCHS = ["resnet50"] if PILOTO else ["resnet50", "densenet121", "efficientnetb0"]


## 1. Sesión

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU"
import sys
print("Python", sys.version.split()[0])
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}
!git rev-parse HEAD
!git status --porcelain
!pip install -q -r requirements.txt


## 2. Huellas

Si alguna no coincide, esta celda para y no se entrena. La huella del zip se imprime; el archivo solo vive en Drive.


In [ ]:
import os, shutil, zipfile, time
from pathlib import Path
from src.tracking import load_config, file_sha256

HUELLAS = {
    "split_v1.csv": "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38",
    "class_weights.json": "965f0301fdfe13fcc3925f1263c2171cd25f49760408261fc39b83b337b04d86",
    "HAM10000_metadata.csv": "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5",
    "src/augmentation.py": "e6649f772196e4090641430e829a0475f72ce5ffcf4d4f0937bf33de4046d72e",
    "configs/m2_v3.yaml": "21b4c8ff053e354d08323592676dfd504c133f87d082df5f50310a74e6e43f15",
}
N_IMAGENES = 10015

cfg = load_config("configs/m2_v3.yaml")
assert cfg["config_version"] == 3, cfg.get("config_version")
assert cfg["training"]["phase2"]["unfreeze_fraction"] == 1.0
assert cfg["training"]["phase2"]["freeze_batchnorm"] is True
assert cfg["data"]["augmentation"]["politica"] == "uniforme"

zip_path = cfg["paths"]["dataset_zip"]
assert Path(zip_path).is_file(), f"No está el zip: {zip_path}"
print("huella zip", file_sha256(zip_path))

LOCAL = Path(cfg["paths"]["local_data_dir"])
if not (LOCAL / "images").exists():
    t0 = time.time()
    shutil.copy(zip_path, "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time() - t0:.0f} s)")

comprobaciones = {
    "split_v1.csv": "data/splits/split_v1.csv",
    "class_weights.json": "data/splits/class_weights.json",
    "HAM10000_metadata.csv": cfg["paths"]["metadata_csv"],
    "src/augmentation.py": "src/augmentation.py",
    "configs/m2_v3.yaml": "configs/m2_v3.yaml",
}
fallos = []
for nombre, ruta in comprobaciones.items():
    huella = file_sha256(ruta)
    if huella != HUELLAS[nombre]:
        fallos.append(f"{nombre}: {huella} != {HUELLAS[nombre]}")
n = len(list((LOCAL / "images").glob("*.jpg")))
if n != N_IMAGENES:
    fallos.append(f"imágenes del zip: {n} != {N_IMAGENES}")
if fallos:
    raise SystemExit("Huella distinta. No se entrena.\n" + "\n".join(fallos))
print("[+] Huellas coinciden. Se puede entrenar.")


In [ ]:
import os
import mlflow
from google.colab import userdata
from src.tracking import setup_mlflow, git_dirty

os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print("MLflow:", setup_mlflow(cfg))
print("git_dirty =", git_dirty(), "| rama", BRANCH, "| semilla", SEED, "|", ARCHS)
assert not git_dirty(), "El clon de la rama tiene cambios. No se entrena."
assert SEED in cfg["training"]["seeds"], SEED
assert set(ARCHS) <= set(cfg["training"]["architectures"])


## 3. Entrenar

Cada arquitectura es un run `m2-<arch>-s<semilla>` en `m2-entrenamiento-v3`. Al terminar registra la prueba de regresión del atajo. Si la alarma salta, el run queda en `estado=final` y el piloto no pasa.


In [ ]:
from pathlib import Path
from src.training import lanzar_run

def run_cerrado(arch):
    id_file = Path(cfg["paths"]["checkpoints_v3_dir"]) / f"m2-{arch}-s{SEED}" / "mlflow_run_id.txt"
    if not id_file.exists():
        return None
    run = mlflow.get_run(id_file.read_text().strip())
    if run.info.status == "FINISHED" and run.data.tags.get("estado") == "final":
        return run
    return None

for arch in ARCHS:
    previo = run_cerrado(arch)
    if previo is not None:
        print(f"{arch}: ya está en estado=final ({previo.info.run_id}). No se repite.")
        continue
    print(f"\n{'=' * 20} {arch} · semilla {SEED} {'=' * 20}")
    try:
        lanzar_run(arch, SEED, cfg, "real", AUTOR)
    except RuntimeError as e:
        if "ya terminó" not in str(e):
            raise
        cerrado = run_cerrado(arch)
        if cerrado is None:
            raise
        print(f"{arch}: el run ya había terminado ({cerrado.info.run_id}).")


## 4. Resumen copiable

In [ ]:
from pathlib import Path

def _fmt(m, clave):
    v = m.get(clave)
    return "—" if v is None else f"{v:.3f}"

lineas = []
alguna_alarma = False
for arch in ARCHS:
    id_file = Path(cfg["paths"]["checkpoints_v3_dir"]) / f"m2-{arch}-s{SEED}" / "mlflow_run_id.txt"
    if not id_file.exists():
        lineas.append(f"{arch}  pendiente")
        continue
    run = mlflow.get_run(id_file.read_text().strip())
    m, p, t = run.data.metrics, run.data.params, run.data.tags
    minutos = m.get("tiempo_total_s")
    minutos_txt = "—" if minutos is None else f"{minutos / 60:.1f}"
    alarma = t.get("alarma_atajo", "—")
    alguna_alarma = alguna_alarma or alarma == "true"
    try:
        epoca = int(p["f1_epocas_ejecutadas"]) + int(p["f2_mejor_epoca"])
    except (KeyError, ValueError):
        epoca = "—"
    lineas.append(
        f"{arch}  run={run.info.run_id}  estado={t.get('estado', '—')}  "
        f"epoca_guardada={epoca}  macro_f1={_fmt(m, 'val_macro_f1')}  "
        f"sens_mel={_fmt(m, 'val_mel_sensitivity')}  auc_mel={_fmt(m, 'val_auc_mel')}  "
        f"frac_nv_limpio={_fmt(m, 'val_frac_nv_limpio')}  "
        f"frac_nv_aumentado={_fmt(m, 'val_frac_nv_aumentado')}  "
        f"params_f2={p.get('params_entrenables_f2', '—')}  "
        f"minutos={minutos_txt}  alarma_atajo={alarma}"
    )

print(f"semilla {SEED}  autor {AUTOR}  experimento m2-entrenamiento-v3  piloto={PILOTO}")
print("\n".join(lineas))
if alguna_alarma:
    print("ALARMA de atajo. El piloto no pasa: no lances el resto de runs.")
else:
    print("Sin alarma de atajo en las arquitecturas ya cerradas.")
